##Eigenvector Preparation Using Quantum Phase Estimation and Grover's Algorithm

##Project Overview

Suppose U is a unitary operator whose eigenvectors are the computational basis states: U∣x⟩=exp(2πiθ(x))∣x⟩. Given a target phase t, our goal is to prepare the basis state ∣x⟩ satisfying θ(x)=t.

Rather than classically searching through all possible basis states, we use:
Quantum Phase Estimation (QPE) to extract phase information,
Grover's algorithm to amplify states having the desired phase.
The final circuit prepares the eigenvector corresponding to the target eigenvalue.

##Necessary Imports

In [ ]:
!pip install qiskit qiskit-aer
import numpy as np
from math import pi, sqrt, asin
from fractions import Fraction
from qiskit_aer import AerSimulator

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import DiagonalGate
from qiskit.quantum_info import Statevector

# Manual Quantum Fourier Transform

The following functions implement the Quantum Fourier Transform and its inverse using Hadamard gates and controlled phase rotations. These routines will later be used inside Quantum Phase Estimation.

In [3]:
def qft(qc, qubits):
    m = len(qubits)
    for j in range(m):
        qc.h(qubits[j])
        for k in range(j + 1, m):
            qc.cp(pi / (2 ** (k - j)), qubits[k], qubits[j])


def inverse_qft(qc, qubits):
    m = len(qubits)
    for j in reversed(range(m)):
        for k in reversed(range(j + 1, m)):
            qc.cp(-pi / (2 ** (k - j)), qubits[k], qubits[j])
        qc.h(qubits[j])

# Quantum Phase Estimation

The function below applies Quantum Phase Estimation to the eigenvector register.

The phase register is first placed into a uniform superposition. Controlled powers of the diagonal unitary are then applied, followed by the inverse Quantum Fourier Transform.

The resulting phase register contains the binary representation of the eigenphase.

In [4]:
def apply_qpe(qc, xreg, preg, phases):
    d = len(preg)

    for q in preg:
        qc.h(q)

    for j in range(d):
        diagonal_entries = [
            np.exp(2j * np.pi * (2 ** j) * float(theta))
            for theta in phases
        ]

        U_power = DiagonalGate(diagonal_entries)
        controlled_U_power = U_power.control(1)

        qc.append(controlled_U_power, [preg[j]] + list(xreg))

    inverse_qft(qc, preg)

# Uncomputing Quantum Phase Estimation

After the phase information has been used by the oracle, the phase register must be restored to its initial state.

The following function exactly reverses the QPE circuit.

In [5]:
def uncompute_qpe(qc, xreg, preg, phases):
    d = len(preg)

    qft(qc, preg)

    for j in reversed(range(d)):
        diagonal_entries = [
            np.exp(-2j * np.pi * (2 ** j) * float(theta))
            for theta in phases
        ]

        U_power_inverse = DiagonalGate(diagonal_entries)
        controlled_U_power_inverse = U_power_inverse.control(1)

        qc.append(controlled_U_power_inverse, [preg[j]] + list(xreg))

    for q in preg:
        qc.h(q)


# Phase Oracle

The phase oracle marks all states whose eigenphase matches the target phase.

The target phase is represented as an integer and compared against the contents of the phase register.

In [6]:
def mark_target_phase(qc, preg, mark, target_int):
    d = len(preg)

    for j in range(d):
        if ((target_int >> j) & 1) == 0:
            qc.x(preg[j])

    qc.mcx(list(preg), mark)

    for j in range(d):
        if ((target_int >> j) & 1) == 0:
            qc.x(preg[j])


# Grover Diffuser

The diffuser performs inversion about the mean on the eigenvector register.

This step amplifies the amplitudes of states marked by the oracle.

In [7]:
def diffuser(qc, xreg):
    n = len(xreg)

    qc.h(xreg)
    qc.x(xreg)

    if n == 1:
        qc.z(xreg[0])
    else:
        qc.h(xreg[-1])
        qc.mcx(list(xreg[:-1]), xreg[-1])
        qc.h(xreg[-1])

    qc.x(xreg)
    qc.h(xreg)


# Complete Eigenvector Preparation Algorithm

The following function combines QPE, phase marking, uncomputation, and Grover amplification to prepare the eigenvector corresponding to a specified eigenphase.

In [8]:
def eigenvector_preparation(phases, d, t):
    """
    Input:
        phases = [theta(0), theta(1), ..., theta(2^n - 1)]
        where U|x> = exp(2*pi*i*theta(x)) |x>.

        d = positive integer such that 2^d theta(x) is an integer.

        t = target phase.

    Output:
        A quantum circuit that prepares the unique |x> such that theta(x) = t.


    """

    N = len(phases)
    n = int(np.log2(N))

    if 2 ** n != N:
        raise ValueError("The number of phases must be a power of 2.")

    target_int = int(round((2 ** d) * float(t))) % (2 ** d)

    x = QuantumRegister(n, "x")
    phase = QuantumRegister(d, "phase")
    mark = QuantumRegister(1, "mark")

    qc = QuantumCircuit(x, phase, mark)

    qc.h(x)

    qc.x(mark[0])
    qc.h(mark[0])

    theta = asin(1 / sqrt(N))
    iterations = int(round(pi / (4 * theta) - 0.5))

    for _ in range(iterations):
        apply_qpe(qc, x, phase, phases)
        mark_target_phase(qc, phase, mark[0], target_int)
        uncompute_qpe(qc, x, phase, phases)
        diffuser(qc, x)

    qc.h(mark[0])
    qc.x(mark[0])

    return qc



# Statevector Verification

The helper function below computes the probability distribution over the eigenvector register. This allows us to verify that the target eigenvector has been successfully amplified.

In [9]:
def x_probabilities(qc, n):
    sv = Statevector.from_instruction(qc)

    probs = {}

    for i, amp in enumerate(sv.data):
        bits = format(i, f"0{qc.num_qubits}b")[::-1]
        x_bits = bits[:n]
        probs[x_bits] = probs.get(x_bits, 0) + abs(amp) ** 2

    return sorted(probs.items(), key=lambda item: item[1], reverse=True)

# Example

In the example below,  since the phase associated with the basis state |10> is 1/2, the algorithm should amplify the state |10>.

In [18]:
phases1 = [
    Fraction(0, 4),
    Fraction(1, 4),
    Fraction(1, 2),
    Fraction(3, 4),
]

qc1 = eigenvector_preparation(phases1, d=2, t=Fraction(1, 2))




print(x_probabilities(qc1, n=2))



[('10', np.float64(0.9999999999999971)), ('01', np.float64(2.139971279804893e-32)), ('00', np.float64(2.1280905951488703e-32)), ('11', np.float64(3.471949705726813e-33))]
